# Week 3 — Numerical Methods for ODEs

> **Differential Equations for Scientists & Engineers**  
> *Euler, Runge-Kutta, Adams-Bashforth — every solver built from first principles.*

---

## Learning Objectives

1. Derive **Taylor-series error analysis** for Euler's method (LTE and GTE)
2. Implement **Midpoint**, **Heun**, and **classical RK4** from scratch
3. Build **multi-step methods**: Adams-Bashforth (explicit) and Adams-Moulton (implicit)
4. Produce **order-convergence plots** to empirically verify theoretical error rates
5. Compare solvers on the Lorenz system and van der Pol oscillator


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

---

## 1. Euler's Method — Derivation and Error Analysis

Consider $y' = f(t, y)$, $y(t_0) = y_0$. Expand $y(t+h)$ in a Taylor series:

$$y(t+h) = y(t) + h\,y'(t) + \frac{h^2}{2}y''(t) + O(h^3)$$

Truncating after the first-order term gives **Euler's method**:

$$\boxed{y_{n+1} = y_n + h\,f(t_n, y_n)}$$

- **Local Truncation Error (LTE):** $\tau_n = O(h^2)$ — error per step
- **Global Truncation Error (GTE):** $e_n = O(h)$ — error at fixed $T$ (first-order method)

In [ ]:
def euler(f, t0, y0, T, h):
    """Forward Euler solver for y' = f(t, y). Returns (t_arr, y_arr)."""
    t_arr = np.arange(t0, T + h/2, h)
    y_arr = np.zeros((len(t_arr), *np.shape(y0)))
    y_arr[0] = y0
    for n in range(len(t_arr) - 1):
        y_arr[n+1] = y_arr[n] + h * np.asarray(f(t_arr[n], y_arr[n]))
    return t_arr, y_arr


def midpoint(f, t0, y0, T, h):
    """Explicit midpoint (RK2) solver."""
    t_arr = np.arange(t0, T + h/2, h)
    y_arr = np.zeros((len(t_arr), *np.shape(y0)))
    y_arr[0] = y0
    for n in range(len(t_arr) - 1):
        yn = y_arr[n]; tn = t_arr[n]
        k1 = np.asarray(f(tn, yn))
        k2 = np.asarray(f(tn + h/2, yn + h/2 * k1))
        y_arr[n+1] = yn + h * k2
    return t_arr, y_arr


def rk4(f, t0, y0, T, h):
    """Classical 4th-order Runge-Kutta solver."""
    t_arr = np.arange(t0, T + h/2, h)
    y_arr = np.zeros((len(t_arr), *np.shape(y0)))
    y_arr[0] = y0
    for n in range(len(t_arr) - 1):
        yn = y_arr[n]; tn = t_arr[n]
        k1 = np.asarray(f(tn,        yn))
        k2 = np.asarray(f(tn + h/2,  yn + h/2 * k1))
        k3 = np.asarray(f(tn + h/2,  yn + h/2 * k2))
        k4 = np.asarray(f(tn + h,    yn + h   * k3))
        y_arr[n+1] = yn + h/6 * (k1 + 2*k2 + 2*k3 + k4)
    return t_arr, y_arr


# ---- Convergence test on y' = -2y, y(0)=1, y_exact = e^(-2t) ----
f_test    = lambda t, y: np.array([-2.0 * y[0]])
y_exact   = lambda T: np.exp(-2.0 * T)
T_final   = 3.0
h_values  = np.array([0.5, 0.25, 0.1, 0.05, 0.025, 0.01])

methods = {
    'Euler (p=1)':    (euler,    1, '#E53935'),
    'Midpoint (p=2)': (midpoint, 2, '#FB8C00'),
    'RK4 (p=4)':      (rk4,      4, '#43A047'),
}

fig, ax = plt.subplots(figsize=(8, 5))
for label, (solver, order, color) in methods.items():
    errors = []
    for h in h_values:
        _, y_arr = solver(f_test, 0.0, np.array([1.0]), T_final, h)
        errors.append(abs(y_arr[-1, 0] - y_exact(T_final)))
    ax.loglog(h_values, errors, 'o-', color=color, lw=2, label=label)

# Reference slopes
h_ref = np.array([h_values[0], h_values[-1]])
for order, ls, alpha in [(1, '--', 0.5), (2, ':', 0.5), (4, '-.', 0.5)]:
    ax.loglog(h_ref, 0.5 * h_ref**order, ls=ls, color='gray', alpha=alpha,
              label=f'$O(h^{order})$ reference')

ax.set_xlabel('Step size h'); ax.set_ylabel('Global error at T=3')
ax.set_title('Convergence Order Verification')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

---

## 2. Runge-Kutta Family — From RK2 to RK4

The **Runge-Kutta** methods avoid storing history (unlike multi-step) by evaluating $f$ at multiple **stages** within each step.

**RK2 (Midpoint method):** Two stages, 2nd-order accurate ($O(h^2)$):
$$k_1 = f(t_n, y_n), \quad k_2 = f\!\left(t_n + \tfrac{h}{2},\, y_n + \tfrac{h}{2}k_1\right)$$
$$y_{n+1} = y_n + h\,k_2$$

**RK4 (Classical):** Four stages, 4th-order accurate ($O(h^4)$):
$$k_1 = f(t_n,y_n),\quad k_2 = f(t_n+\tfrac{h}{2}, y_n+\tfrac{h}{2}k_1)$$
$$k_3 = f(t_n+\tfrac{h}{2}, y_n+\tfrac{h}{2}k_2),\quad k_4 = f(t_n+h, y_n+hk_3)$$
$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

The **Butcher tableau** is a compact notation for any RK scheme:
$$\begin{array}{c|cc}0 & \\ 1/2 & 1/2 & \\ \hline & 0 & 1\end{array}\quad\text{(RK2 Midpoint)}$$

**Convergence:** global error $E \propto h^p$ where $p$ is the order — confirmed empirically by doubling $h$ and measuring $E$.

In [ ]:
def rk2_midpoint(f, t0, y0, T, h):
    """Explicit midpoint (RK2) method: 2nd-order Runge-Kutta."""
    t_arr = np.arange(t0, T + h/2, h)
    y_arr = np.zeros((len(t_arr), *np.shape(y0)))
    y_arr[0] = y0
    for n in range(len(t_arr) - 1):
        k1 = np.asarray(f(t_arr[n], y_arr[n]))
        k2 = np.asarray(f(t_arr[n] + h/2, y_arr[n] + h/2 * k1))
        y_arr[n+1] = y_arr[n] + h * k2
    return t_arr, y_arr


def rk4(f, t0, y0, T, h):
    """Classical RK4 — 4th-order Runge-Kutta."""
    t_arr = np.arange(t0, T + h/2, h)
    y_arr = np.zeros((len(t_arr), *np.shape(y0)))
    y_arr[0] = y0
    for n in range(len(t_arr) - 1):
        k1 = np.asarray(f(t_arr[n],       y_arr[n]))
        k2 = np.asarray(f(t_arr[n]+h/2,   y_arr[n]+h/2*k1))
        k3 = np.asarray(f(t_arr[n]+h/2,   y_arr[n]+h/2*k2))
        k4 = np.asarray(f(t_arr[n]+h,     y_arr[n]+h*k3))
        y_arr[n+1] = y_arr[n] + h*(k1 + 2*k2 + 2*k3 + k4)/6
    return t_arr, y_arr


# ── Convergence test on y' = -y, exact: y = e^{-t} ─────────────────────────
def f_test(t, y): return -y
y0_test, T_test = np.array([1.0]), 5.0
y_exact_fn = lambda t: np.exp(-t)

h_values = [0.5, 0.25, 0.1, 0.05, 0.02, 0.01]
methods = {
    'Euler (p=1)':    (euler,       '#e74c3c'),
    'RK2 (p=2)':      (rk2_midpoint,'#f39c12'),
    'RK4 (p=4)':      (rk4,         '#27ae60'),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax_conv = axes[0]

for label, (method, color) in methods.items():
    errors = []
    for h in h_values:
        t_arr, y_arr = method(f_test, 0, y0_test, T_test, h)
        err = np.max(np.abs(y_arr[:,0] - y_exact_fn(t_arr)))
        errors.append(err)
    ax_conv.loglog(h_values, errors, 'o-', color=color, lw=2, label=label)

# Reference slopes
h_ref = np.array([0.02, 0.5])
ax_conv.loglog(h_ref, 0.5*h_ref**1, 'k--', lw=1, alpha=0.5, label='slope 1')
ax_conv.loglog(h_ref, 0.3*h_ref**2, 'k:',  lw=1, alpha=0.5, label='slope 2')
ax_conv.loglog(h_ref, 0.05*h_ref**4,'k-.', lw=1, alpha=0.5, label='slope 4')
ax_conv.set_xlabel('Step size $h$'); ax_conv.set_ylabel('Max global error')
ax_conv.set_title('Convergence: Euler vs RK2 vs RK4')
ax_conv.legend(fontsize=9); ax_conv.grid(alpha=0.3, which='both')

# Side-by-side solutions for stiff-ish problem
ax_sol = axes[1]
h_coarse = 0.3
t_e, y_e   = euler(f_test, 0, np.array([1.0]), 5.0, h_coarse)
t_r2, y_r2 = rk2_midpoint(f_test, 0, np.array([1.0]), 5.0, h_coarse)
t_r4, y_r4 = rk4(f_test, 0, np.array([1.0]), 5.0, h_coarse)
t_fine = np.linspace(0, 5, 400)
ax_sol.plot(t_fine, np.exp(-t_fine), 'k-', lw=2, label='Exact $e^{-t}$')
ax_sol.plot(t_e,  y_e[:,0],  'o--', color='#e74c3c', label=f'Euler h={h_coarse}')
ax_sol.plot(t_r2, y_r2[:,0], 's--', color='#f39c12', label=f'RK2 h={h_coarse}')
ax_sol.plot(t_r4, y_r4[:,0], '^--', color='#27ae60', label=f'RK4 h={h_coarse}')
ax_sol.set_xlabel('t'); ax_sol.set_ylabel('y')
ax_sol.set_title(f"Solutions at coarse $h={h_coarse}$")
ax_sol.legend(fontsize=9); ax_sol.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 3. Multi-Step Methods — Adams Family

Single-step methods recompute all function values at each step.  
**Multi-step** methods recycle past evaluations:

**Adams-Bashforth 4-step (explicit, order 4):**

$$y_{n+1} = y_n + \frac{h}{24}\bigl(55f_n - 59f_{n-1} + 37f_{n-2} - 9f_{n-3}\bigr)$$

**Adams-Moulton 3-step (implicit, order 4):**

$$y_{n+1} = y_n + \frac{h}{24}\bigl(9f_{n+1} + 19f_n - 5f_{n-1} + f_{n-2}\bigr)$$

A **predictor-corrector** pair (AB4-AM4) evaluates $f$ twice per step instead of four times for RK4, at the same order.

In [ ]:
def adams_bashforth4(f, t0, y0, T, h):
    """4-step Adams-Bashforth. Uses RK4 to generate the first 3 steps."""
    # Startup with RK4
    t_arr = [t0]
    y_arr = [np.asarray(y0, dtype=float)]
    f_arr = [np.asarray(f(t0, y_arr[0]))]

    for _ in range(3):
        yn = y_arr[-1]; tn = t_arr[-1]
        k1 = np.asarray(f(tn,        yn))
        k2 = np.asarray(f(tn + h/2,  yn + h/2*k1))
        k3 = np.asarray(f(tn + h/2,  yn + h/2*k2))
        k4 = np.asarray(f(tn + h,    yn + h*k3))
        y_next = yn + h/6*(k1 + 2*k2 + 2*k3 + k4)
        t_next = tn + h
        t_arr.append(t_next); y_arr.append(y_next)
        f_arr.append(np.asarray(f(t_next, y_next)))

    # AB4 main loop
    while t_arr[-1] < T - h/2:
        fn   = f_arr[-1]; fn1 = f_arr[-2]; fn2 = f_arr[-3]; fn3 = f_arr[-4]
        y_next = y_arr[-1] + h/24 * (55*fn - 59*fn1 + 37*fn2 - 9*fn3)
        t_next = t_arr[-1] + h
        t_arr.append(t_next); y_arr.append(y_next)
        f_arr.append(np.asarray(f(t_next, y_next)))

    return np.array(t_arr), np.array(y_arr)


# Compare AB4 with RK4 on y' = cos(t) - y, exact: y = 0.5*(cos t + sin t + e^{-t})
f_cos = lambda t, y: np.array([np.cos(t) - y[0]])
y_exact_cos = lambda t: 0.5 * (np.cos(t) + np.sin(t) + np.exp(-t))

T = 8.0; h = 0.1
t_rk4, y_rk4 = rk4(f_cos, 0, np.array([1.0]), T, h)
t_ab4,  y_ab4  = adams_bashforth4(f_cos, 0, np.array([1.0]), T, h)
t_ex = np.linspace(0, T, 800)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t_ex, y_exact_cos(t_ex), 'k-', lw=2, label='Exact')
axes[0].plot(t_rk4, y_rk4[:, 0], 'b--', lw=1.5, label='RK4')
axes[0].plot(t_ab4, y_ab4[:, 0], 'r:', lw=2, label='AB4')
axes[0].legend(frameon=False); axes[0].set_title("$y' = \\cos t - y$")

axes[1].semilogy(t_rk4, np.abs(y_rk4[:, 0] - y_exact_cos(t_rk4)), 'b-', label='RK4 error')
axes[1].semilogy(t_ab4, np.abs(y_ab4[:, 0] - y_exact_cos(t_ab4)), 'r-', label='AB4 error')
axes[1].legend(frameon=False); axes[1].set_title('Absolute Error Comparison')

plt.tight_layout(); plt.show()

---

## 4. Application — Lorenz System

The Lorenz system is the canonical example of **deterministic chaos**:

$$\dot{x} = \sigma(y - x), \quad \dot{y} = x(\rho - z) - y, \quad \dot{z} = xy - \beta z$$

Classical parameters: $\sigma = 10$, $\rho = 28$, $\beta = 8/3$.

In [ ]:
def lorenz(t, state, sigma=10., rho=28., beta=8./3.):
    x, y, z = state
    return np.array([
        sigma * (y - x),
        x * (rho - z) - y,
        x * y - beta * z
    ])

_, xyz = rk4(lorenz, 0, np.array([1., 1., 1.]), 50, 0.01)
skip = 500  # discard transient

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

N = len(xyz[skip:])
colors = cm.plasma(np.linspace(0, 1, N))
for i in range(N - 1):
    ax.plot(xyz[skip+i:skip+i+2, 0],
            xyz[skip+i:skip+i+2, 1],
            xyz[skip+i:skip+i+2, 2],
            color=colors[i], lw=0.4, alpha=0.8)

ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Lorenz Attractor — RK4 ($h=0.01$)')
plt.tight_layout(); plt.show()

---

## 5. Exercises

1. **(Euler error)** For $y' = \lambda y$, $y(0)=1$, derive analytically the global error at $T=1$ as a function of $h$ and $\lambda$. Verify your formula numerically for $\lambda = -1, -5, -20$.

2. **(Heun's method)** Implement Heun's method (trapezoidal predictor-corrector, order 2) and add it to the convergence plot. Confirm its slope.

3. **(Butcher tableau)** Implement the **3/8-rule RK4** using the generic Butcher-tableau solver and verify it gives the same order as classical RK4.

4. **(Adams-Moulton)** Implement the AB4-AM4 predictor-corrector pair. Compare its accuracy and cost (function evaluations) to RK4 on the Lorenz system.

5. **(Chaos)** Run two Lorenz trajectories with initial conditions differing by $10^{-8}$ in one component. Plot the distance between them over time and estimate the **Lyapunov exponent**.